# Fine-tune the ScholarCrew writer with LoRA (Kaggle)

Trains a small adapter on top of Qwen3-8B, using the checked examples built by `scholarcrew.finetune.build_dataset`.

**Before you start, in the right-hand panel:**
1. Settings > **Accelerator** > **GPU T4 x2** (choose T4, not P100).
2. Settings > **Internet** > **On** (needs a phone-verified Kaggle account).
3. **Add Input** > **Upload** > create a **private** dataset containing your `train.jsonl`.

Then run the cells from top to bottom.

In [ ]:
# 1. Check the GPU. You should see two Tesla T4 cards.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the code and the libraries
%cd /kaggle/working
!git clone -q https://github.com/omsawant-ds/scholarcrew.git
%cd scholarcrew
!pip install -q -U transformers peft accelerate bitsandbytes

In [ ]:
# 3. Find your uploaded training examples and put them where the script expects them.
import glob
import os
import shutil

found = glob.glob('/kaggle/input/**/train.jsonl', recursive=True)
assert found, 'train.jsonl not found. Add your private dataset with Add Input > Upload.'
os.makedirs('data/sft', exist_ok=True)
shutil.copy(found[0], 'data/sft/train.jsonl')
print('using', found[0])
!wc -l data/sft/train.jsonl

In [ ]:
# 4. Start the training in the BACKGROUND. It keeps running if the browser disconnects or the
#    kernel dies, and it saves its progress every 3 steps. About 75 to 90 minutes for Qwen3-4B.
#    If the session restarts, run this cell again: it resumes from the last saved step.
#    (Remove '--fresh' from the list below to resume instead of starting over.)
import os
import subprocess
import time

# Stop any earlier training still holding the GPU. Two runs at once share the same 15 GB card
# and both run out of memory.
subprocess.run(['pkill', '-9', '-f', 'train_lora'], check=False)
time.sleep(4)

env = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES='0',
    HF_HOME='/kaggle/temp/hf',
    PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
    PYTHONPATH='src',
)
command = [
    'python', '-u', '-m', 'scholarcrew.finetune.train_lora',
    '--model', 'Qwen/Qwen3-4B', '--out', 'outputs/writer-lora', '--fresh',
]
with open('/kaggle/working/train.log', 'w') as log:  # the background process keeps its own copy
    process = subprocess.Popen(
        command, cwd='/kaggle/working/scholarcrew', env=env,
        stdout=log, stderr=subprocess.STDOUT,
        start_new_session=True,  # keeps running even if this notebook's kernel dies
    )
print('started, process id', process.pid, '- run the next cell to watch it')

In [ ]:
# 4b. Watch the training. Run this cell as often as you like.
#     Look for 'loss' lines (one per step) and, at the end, 'Saved the adapter'.
!tail -n 15 /kaggle/working/train.log
!echo
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader
!pgrep -f train_lora > /dev/null && echo 'STATUS: running' || echo 'STATUS: finished or stopped'

In [ ]:
%cd /kaggle/working/scholarcrew
# 5. Package the adapter so it appears under Output and can be downloaded.
import os

assert os.path.exists('outputs/writer-lora/adapter_model.safetensors'), 'Training did not finish: there is no adapter to download. Run cell 4b to read the training log.'
!cd outputs && zip -qr /kaggle/working/writer-lora.zip writer-lora -x 'writer-lora/checkpoints/*'
!ls -lh /kaggle/working/writer-lora.zip

**To download the adapter:** open the right-hand panel > **Data** > **Output** > `/kaggle/working/writer-lora.zip` > the three dots > **Download**.

If you close the notebook first, use **Save Version > Save & Run All** instead, and download the zip from the version's **Output** tab.